# 3단계 · 베이스라인 (기준선 만들기)

**왜 하나요?** "내 모델 정확도 70%!"는 그 자체로는 의미가 없습니다. 이 문제는 **무조건 `of`로 찍어도 약 54%**가 나오기 때문이에요. 그래서 먼저 *기준선*을 재두고, 나중에 BERT가 이걸 얼마나 넘는지로 **개선 폭**을 증명합니다.

이 노트북에서 만드는 기준선 2개:
1. **다수 클래스(majority)** — 항상 가장 흔한 `of`로 찍기 → 성능의 *바닥*
2. **TF-IDF + 로지스틱 회귀** — 딥러닝 없이 *단어 통계*만으로 분류하는 고전적 방법 → 의미 있는 비교점

**평가 데이터:** 검증셋(val)으로 측정합니다. **테스트셋(test)은 맨 마지막 5단계까지 손대지 않습니다.**

**평가지표 2개**
- **정확도(accuracy)** — 전체 중 몇 개를 맞혔나 (직관적이지만 불균형엔 약함)
- **Macro-F1** — of/from/with **각 클래스의 성적을 평등하게 평균**낸 점수. 소수 클래스(from/with)를 무시하지 못하게 하는, 불균형 데이터의 핵심 지표.

In [1]:
# pandas: 표(CSV) 데이터를 읽고 다루는 도구
import pandas as pd

# DummyClassifier: 실제 학습 없이 단순 규칙(예: 항상 다수 클래스)으로 예측하는 '바닥' 비교용 모델
from sklearn.dummy import DummyClassifier

# TfidfVectorizer: 문장을 '어떤 단어가 얼마나 중요하게 등장하는지'의 숫자 표로 바꿔주는 도구
from sklearn.feature_extraction.text import TfidfVectorizer

# LogisticRegression: 그 숫자 표를 보고 of/from/with를 분류하는 고전적 통계 모델
from sklearn.linear_model import LogisticRegression

# Pipeline: 여러 단계를 하나로 묶어 '문장 -> 숫자변환 -> 분류'를 한 번에 처리
from sklearn.pipeline import Pipeline

# 평가지표들: 정확도, macro-F1, 클래스별 상세표, 혼동행렬
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [2]:
# 2단계에서 저장한 분할 파일을 불러옵니다.
train_df = pd.read_csv("../data/splits/train.csv")  # 학습용 (모델이 공부)
val_df   = pd.read_csv("../data/splits/val.csv")    # 검증용 (성능 측정)

print("학습:", len(train_df), "검증:", len(val_df))
train_df.head(3)

학습: 657 검증: 141


,id,label,product,material,sentence,sentence_masked
0,313,of,The shaft,"bone , hollowed @ @ @ @ @ @ @ @ @ @","The shaft , she thinks , is made of bone , hol...","The shaft , she thinks , is made ___ bone , ho..."
1,362,of,whiskey,soured corn,They passed around a half-full bottle of whisk...,They passed around a half-full bottle of whisk...
2,768,with,FRSCCM,fiber factors higher than Fsub d,For FRSCCM made with fiber factors higher than...,For FRSCCM made ___ fiber factors higher than ...


### 입력으로 무엇을 줄까?

`sentence_masked`(정답 전치사를 `made ___`로 가린 문장)를 **입력**으로, `label`(of/from/with)을 **정답**으로 씁니다.

전치사를 가린 이유: 가리지 않으면 모델이 문장 속 `made of`를 그대로 베껴서 100% 맞히는 **부정행위(정답 누출)**가 일어나기 때문입니다. 이 입력 형태는 **베이스라인과 BERT 모두 동일하게** 써서 비교가 공정하도록 합니다.

In [3]:
# X = 입력(가려진 문장), y = 정답(전치사). 머신러닝 관례로 입력은 대문자 X, 정답은 소문자 y.
X_train, y_train = train_df["sentence_masked"], train_df["label"]
X_val,   y_val   = val_df["sentence_masked"],   val_df["label"]

# 예시 하나로 입력/정답이 어떻게 생겼는지 확인
print("입력 예시:", X_train.iloc[0])
print("정답 예시:", y_train.iloc[0])

입력 예시: The shaft , she thinks , is made ___ bone , hollowed @ @ @ @ @ @ @ @ @ @ .
정답 예시: of


## 기준선 ① 다수 클래스 (항상 `of`로 찍기)

가장 단순한 바닥. 학습 데이터에서 제일 흔한 클래스를 찾아, 검증셋의 모든 문장에 그 답을 찍습니다.

In [4]:
# strategy="most_frequent": 학습 데이터에서 가장 많은 클래스(여기선 of)를 항상 예측
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)              # '가장 흔한 클래스'가 무엇인지만 학습
pred_dummy = dummy.predict(X_val)        # 검증셋 전체에 그 답을 찍음

# 정확도와 macro-F1 계산
acc_dummy = accuracy_score(y_val, pred_dummy)
f1_dummy  = f1_score(y_val, pred_dummy, average="macro")  # average='macro': 클래스별 F1을 평등 평균
print(f"[다수 클래스] 정확도={acc_dummy:.3f},  Macro-F1={f1_dummy:.3f}")

[다수 클래스] 정확도=0.546,  Macro-F1=0.235


> `of`가 검증셋의 약 55%라 정확도는 0.55 근처로 나옵니다. 하지만 from/with는 한 번도 못 맞히므로 **Macro-F1은 매우 낮습니다(약 0.23)**. 정확도만 보면 속는 이유가 바로 이것 — 그래서 Macro-F1을 함께 봅니다.

## 기준선 ② TF-IDF + 로지스틱 회귀

딥러닝 없이 **단어 통계**만으로 분류하는 고전적 방법입니다.
- **TF-IDF**: 문장을 '어떤 단어가 얼마나 자주/특징적으로 나오는지'의 숫자표로 변환
- **로지스틱 회귀**: 그 숫자표를 보고 of/from/with 확률을 계산해 가장 높은 걸 선택

`class_weight="balanced"`로 소수 클래스(from/with)에 가중치를 더 줘서 불균형에 대응합니다.

In [5]:
# 두 단계를 하나로 묶은 파이프라인을 만듭니다: 문장 -> TF-IDF 숫자표 -> 로지스틱 회귀 분류
logreg = Pipeline([
    # 1~2개 단어 묶음(ngram_range)을 보고, 너무 드문 단어(min_df=2 미만)는 무시해 잡음을 줄임
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    # max_iter=1000: 충분히 학습하도록 반복 횟수 넉넉히. class_weight='balanced': 불균형 보정
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

logreg.fit(X_train, y_train)        # 학습 데이터로 단어 패턴과 분류 규칙을 학습
pred_lr = logreg.predict(X_val)     # 검증셋을 예측

acc_lr = accuracy_score(y_val, pred_lr)
f1_lr  = f1_score(y_val, pred_lr, average="macro")
print(f"[TF-IDF+LogReg] 정확도={acc_lr:.3f},  Macro-F1={f1_lr:.3f}")

[TF-IDF+LogReg] 정확도=0.560,  Macro-F1=0.525


In [6]:
# 클래스별 상세 성적표: 각 전치사의 정밀도(precision)/재현율(recall)/F1을 보여줍니다.
# (논문에서도 made of/from/with 각각의 precision·recall을 봤던 것과 같은 관점)
print(classification_report(y_val, pred_lr, digits=3))

              precision    recall  f1-score   support

        from      0.362     0.459     0.405        37
          of      0.654     0.662     0.658        77
        with      0.688     0.407     0.512        27

    accuracy                          0.560       141
   macro avg      0.568     0.510     0.525       141
weighted avg      0.584     0.560     0.564       141



In [7]:
# 혼동행렬: 모델이 '무엇을 무엇으로 헷갈렸는지' 보여주는 표.
# 행=실제 정답, 열=모델 예측. 대각선이 정답, 나머지가 혼동.
labels = ["of", "from", "with"]  # 표의 행/열 순서를 고정
cm = confusion_matrix(y_val, pred_lr, labels=labels)
cm_table = pd.DataFrame(cm, index=[f"실제 {l}" for l in labels], columns=[f"예측 {l}" for l in labels])
cm_table

,예측 of,예측 from,예측 with
실제 of,51,23,3
실제 from,18,17,2
실제 with,9,7,11


## 정리 — BERT가 넘어야 할 목표치

| 기준선 | 정확도 | Macro-F1 |
|--------|:------:|:--------:|
| ① 다수 클래스 (of로 찍기) | ~0.55 | ~0.23 |
| ② TF-IDF + 로지스틱 회귀 | (위 출력 참고) | (위 출력 참고) |

다음 4단계에서 BERT를 파인튜닝하면, **②번 점수를 넘어서야** "딥러닝/트랜스포머를 쓴 보람이 있었다"고 말할 수 있습니다. (참고: 논문에서 GPT-4o는 전체 39% 정확도였지만, 데이터 분할·입력 방식이 달라 직접 비교가 아닌 *동기 부여용* 수치입니다.)